# Homework Submission - Stage 04: Data Acquisition and Ingestion

This submission starts from the official starter and completes both required ingestion paths: a public market-data API request and a resilient table scrape. All raw files are timestamped and saved under `data/raw/`.

In [1]:
# Packages used: requests, pandas, python-dotenv, beautifulsoup4
from pathlib import Path
import datetime as dt
import os
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(ROOT / '.env')
TIMEOUT = int(os.getenv('API_TIMEOUT_SECONDS', '30'))
RUN_TS = dt.datetime.now().strftime('%Y%m%d-%H%M%S')

required_files = ['.env', '.env.example', '.gitignore']
file_check = {name: (ROOT / name).exists() for name in required_files}
assert all(file_check.values()), file_check
assert '.env' in {line.strip() for line in (ROOT / '.gitignore').read_text().splitlines()}
print('Setup checks:', file_check)
print('.env is covered by .gitignore: PASS')

Setup checks: {'.env': True, '.env.example': True, '.gitignore': True}
.env is covered by .gitignore: PASS


## 1. API pull - AAPL daily prices

**Source:** `https://query1.finance.yahoo.com/v8/finance/chart/AAPL`

**Parameters:** `range=3mo`, `interval=1d`, `events=history`. The endpoint does not require an API key. The request uses a descriptive user agent, a timeout, status checking, JSON-structure checking, and clear failures.

In [2]:
SYMBOL = 'AAPL'
API_URL = f'https://query1.finance.yahoo.com/v8/finance/chart/{SYMBOL}'
API_PARAMS = {'range': '3mo', 'interval': '1d', 'events': 'history'}
HEADERS = {'User-Agent': 'NYU-AFE-Bootcamp-Homework/1.0 (educational use)'}

response = requests.get(API_URL, params=API_PARAMS, headers=HEADERS, timeout=TIMEOUT)
response.raise_for_status()
payload = response.json()
chart_error = payload.get('chart', {}).get('error')
results = payload.get('chart', {}).get('result')
if chart_error or not results:
    raise RuntimeError(f'Yahoo chart endpoint returned no usable result: {chart_error}')

result = results[0]
quote = result['indicators']['quote'][0]
df_api = pd.DataFrame({
    'date': pd.to_datetime(result['timestamp'], unit='s', utc=True).tz_convert(None),
    'open': quote['open'],
    'high': quote['high'],
    'low': quote['low'],
    'close': quote['close'],
    'volume': quote['volume'],
})
numeric_columns = ['open', 'high', 'low', 'close', 'volume']
df_api[numeric_columns] = df_api[numeric_columns].apply(pd.to_numeric, errors='coerce')
df_api = df_api.sort_values('date').reset_index(drop=True)
df_api.head()

,date,open,high,low,close,volume
0,2026-05-27 13:30:00,308.329987,313.260010,308.299988,310.850006,50430900
1,2026-05-28 13:30:00,310.679993,312.799988,309.570007,312.510010,48220400
2,2026-05-29 13:30:00,311.779999,315.000000,309.529999,312.059998,70026800
3,2026-06-01 13:30:00,309.630005,310.940002,305.019989,306.309998,48849900
4,2026-06-02 13:30:00,307.459991,315.450012,306.690002,315.200012,44534700


In [3]:
def validate_market_data(frame: pd.DataFrame) -> pd.Series:
    required = ['date', 'open', 'high', 'low', 'close', 'volume']
    checks = {
        'required_columns_present': set(required).issubset(frame.columns),
        'rows_positive': len(frame) > 0,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(frame['date']),
        'numeric_types_valid': all(pd.api.types.is_numeric_dtype(frame[c]) for c in required[1:]),
        'required_na_count': int(frame[required].isna().sum().sum()),
        'duplicate_date_count': int(frame['date'].duplicated().sum()),
        'nonpositive_close_count': int((frame['close'].dropna() <= 0).sum()),
        'high_low_rule_violations': int((frame['high'] < frame['low']).sum()),
        'shape': frame.shape,
    }
    return pd.Series(checks, name='API validation')

api_validation = validate_market_data(df_api)
display(api_validation.to_frame())
assert api_validation['required_columns_present']
assert api_validation['rows_positive']
assert api_validation['required_na_count'] == 0
assert api_validation['duplicate_date_count'] == 0
assert api_validation['nonpositive_close_count'] == 0
assert api_validation['high_low_rule_violations'] == 0

api_path = RAW / f'api_yahoo-chart_{SYMBOL}_{RUN_TS}.csv'
df_api.to_csv(api_path, index=False)
print('Saved:', api_path)

,API validation
required_columns_present,True
rows_positive,True
date_is_datetime,True
numeric_types_valid,True
required_na_count,0
duplicate_date_count,0
nonpositive_close_count,0
high_low_rule_violations,0
shape,"(65, 6)"


Saved: /Users/zhangyuang/Desktop/py_file/bootcamp/homework/homework04/data/raw/api_yahoo-chart_AAPL_20260827-104459.csv


## 2. Scrape a permitted public table

**Source:** `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`

**Table:** current S&P 500 constituents. The parser first looks for the stable table id `constituents`, then falls back to the first `wikitable`. It derives headers from `<th>` cells and rejects rows whose width does not match the header.

In [4]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
scrape_response = requests.get(SCRAPE_URL, headers=HEADERS, timeout=TIMEOUT)
scrape_response.raise_for_status()
soup = BeautifulSoup(scrape_response.text, 'html.parser')
table = soup.select_one('table#constituents') or soup.select_one('table.wikitable')
if table is None:
    raise RuntimeError('Could not find the constituents table with the expected selectors.')

header_cells = table.select('thead tr th') or table.select('tr')[0].find_all('th')
headers = [re.sub(r'\[.*?\]', '', cell.get_text(' ', strip=True)).strip() for cell in header_cells]
rows = []
for row in table.select('tbody tr'):
    values = [re.sub(r'\[.*?\]', '', cell.get_text(' ', strip=True)).strip() for cell in row.find_all('td')]
    if len(values) == len(headers):
        rows.append(values)
df_scrape = pd.DataFrame(rows, columns=headers)
df_scrape['CIK'] = pd.to_numeric(df_scrape['CIK'], errors='coerce').astype('Int64')
df_scrape['Date added'] = pd.to_datetime(df_scrape['Date added'], errors='coerce')
df_scrape.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,1467373,1989


In [5]:
def validate_constituents(frame: pd.DataFrame) -> pd.Series:
    required = ['Symbol', 'Security', 'GICS Sector', 'CIK']
    checks = {
        'required_columns_present': set(required).issubset(frame.columns),
        'row_count_at_least_490': len(frame) >= 490,
        'required_na_count': int(frame[required].isna().sum().sum()),
        'duplicate_symbol_count': int(frame['Symbol'].duplicated().sum()),
        'cik_is_numeric': pd.api.types.is_integer_dtype(frame['CIK']),
        'blank_security_count': int(frame['Security'].str.strip().eq('').sum()),
        'shape': frame.shape,
    }
    return pd.Series(checks, name='Scrape validation')

scrape_validation = validate_constituents(df_scrape)
display(scrape_validation.to_frame())
assert scrape_validation['required_columns_present']
assert scrape_validation['row_count_at_least_490']
assert scrape_validation['required_na_count'] == 0
assert scrape_validation['duplicate_symbol_count'] == 0
assert scrape_validation['cik_is_numeric']
assert scrape_validation['blank_security_count'] == 0

scrape_path = RAW / f'scrape_wikipedia_sp500-constituents_{RUN_TS}.csv'
df_scrape.to_csv(scrape_path, index=False)
print('Saved:', scrape_path)

,Scrape validation
required_columns_present,True
row_count_at_least_490,True
required_na_count,0
duplicate_symbol_count,0
cik_is_numeric,True
blank_security_count,0
shape,"(503, 8)"


Saved: /Users/zhangyuang/Desktop/py_file/bootcamp/homework/homework04/data/raw/scrape_wikipedia_sp500-constituents_20260827-104459.csv


## Documentation, assumptions, and risks

- The API file is a **raw acquisition snapshot**, not an adjusted-return research dataset. Prices can later be revised, and exchange timestamps are converted from Unix UTC timestamps to timezone-naive datetimes for portable CSV storage.
- Yahoo and Wikipedia can change schemas, field names, throttling rules, or availability. The validation assertions deliberately fail loudly instead of silently saving malformed data.
- The Wikipedia table is a current-membership list and therefore has survivorship bias if used for historical backtests. It is appropriate here only as a small ingestion exercise.
- HTML selectors are fragile. The id-first/fallback selector and row-width check reduce, but do not eliminate, scrape breakage.
- `.env` exists locally, `.env.example` documents configuration, and `.gitignore` excludes `.env`.

**Result:** both datasets passed required-column, type, missingness, duplicate, shape, and basic-domain checks before they were saved.